# Making SIMSOPT GPU native: augmented Lagrangian

Select **Runtime > Change runtime type > GPU**, then run all cells. This production-scale workflow compares the SIMSOPT CPU oracle with the GPU-native equality-to-zero augmented Lagrangian used by the supplied `auglag_qa.py` approach. It retains flux plus length as the base objective, treats four nonnegative engineering penalties as zero-target constraints, records every multiplier and penalty update, and exports final VTS/VTU files. Failed gates remain valid evidence and do not prevent download.

In [ ]:
import subprocess
subprocess.run(["nvidia-smi"], check=True)

In [ ]:
import importlib
import os
import sys
from pathlib import Path

repo = Path("/content/simsopt")
if not repo.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "gpu-native-objective", "https://github.com/PedroFranciscoGil/simsopt.git", str(repo)], check=True)
else:
    subprocess.run(["git", "fetch", "origin", "gpu-native-objective"], cwd=repo, check=True)
    subprocess.run(["git", "switch", "gpu-native-objective"], cwd=repo, check=True)
    subprocess.run(["git", "pull", "--ff-only"], cwd=repo, check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".", "pytest", "pyevtk"], cwd=repo, check=True)
os.chdir(repo)
source_root = repo / "src"
sys.path.insert(0, str(source_root))
for module_name in tuple(sys.modules):
    if module_name == "simsopt" or module_name.startswith("simsopt."):
        del sys.modules[module_name]
importlib.invalidate_caches()
revision = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo, text=True).strip()
print(revision)

In [ ]:
import jax
import simsopt
from simsopt.gpu import backend_report

resolved_package = Path(simsopt.__file__).resolve()
print(f"Imported SIMSOPT from {resolved_package}")
assert source_root in resolved_package.parents, resolved_package
report = backend_report()
print(report)
assert jax.default_backend() == "gpu", report

In [ ]:
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/gpu"], cwd=repo, check=True)

In [ ]:
import shutil

artifact_root = Path("/content/simsopt-augmented-lagrangian")
if artifact_root.exists():
    shutil.rmtree(artifact_root)
artifact_root.mkdir()
result_file = artifact_root / "stress-augmented-lagrangian.json"
env = os.environ.copy()
env["OMP_NUM_THREADS"] = "1"
env["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
subprocess.run([sys.executable, "benchmarks/gpu/benchmark_augmented_lagrangian.py", "--problem", "stress", "--max-outer-iterations", "8", "--max-inner-iterations", "50", "--mu-init", "10", "--tau", "10", "--mu-max", "1e12", "--current-scale", "100000", "--target-tile-size", "1024", "--source-tile-size", "4320", "--output", str(result_file)], cwd=repo, env=env, check=True)

In [ ]:
import json

result = json.loads(result_file.read_text())
assert result["schema_version"] == 5
assert result["method"]["name"] == "equality_zero_penalty_augmented_lagrangian"
assert result["acceptance_gates"]["gpu_backend"]["passed"]
expected_files = ["stress-augmented-lagrangian-cpu_final_surface.vts", "stress-augmented-lagrangian-cpu_final_coils.vtu", "stress-augmented-lagrangian-gpu_final_surface.vts", "stress-augmented-lagrangian-gpu_final_coils.vtu"]
for filename in expected_files:
    path = artifact_root / filename
    assert path.is_file() and path.stat().st_size > 0, path
decision = {"acceptance_gates": result["acceptance_gates"], "speedup": result["comparison"]["optimization_speedup"], "cpu_final_constraints": result["cpu"]["optimization"]["final_constraints"], "gpu_final_constraints": result["gpu"]["optimization"]["final_constraints"], "cpu_final_metrics": result["cpu"]["final_metrics"], "gpu_final_metrics": result["gpu"]["final_metrics"]}
print(json.dumps(decision, indent=2))

## Interpretation and ParaView

The zero-penalty constraint gate and the physical engineering-feasibility gate answer different questions; inspect both. Open each final `*_surface.vts` in ParaView, choose **Surface**, color by `B_dot_n_over_abs_B` or `abs_B_dot_n_over_abs_B`, and add the matching `*_coils.vtu`. Download the archive unchanged even when a gate fails.

In [ ]:
from google.colab import files
archive = shutil.make_archive("/content/simsopt-augmented-lagrangian", "zip", artifact_root)
files.download(archive)